# StyleGAN and StyleGAN2

## Theory, mathematics, and implementation context

StyleGAN maps $z\sim\mathcal N(0,I)$ through an MLP $w=f(z)$ and injects transformed styles into synthesis layers. A learned constant initializes synthesis; per-layer styles control feature statistics at different scales, while spatial noise supplies stochastic detail. Style mixing trains layer ranges to accept different latent sources. The intermediate W space can make edits more linear than raw Z, though disentanglement is not guaranteed.

StyleGAN2 replaces AdaIN-heavy synthesis with weight modulation and demodulation. Given convolution weights $w_{o,i,k}$ and style scales $s_i$, modulated weights are $w'_{o,i,k}=s_iw_{o,i,k}$; demodulation multiplies output channel $o$ by $d_o=(\sum_{i,k}(w'_{o,i,k})^2+\epsilon)^{-1/2}$. Path length regularization penalizes variation in $\|J_w^Ty\|$, encouraging consistent image sensitivity to W-space movement. This compact 32px CIFAR-10 implementation teaches these operations; it is not the fused, high-resolution official training system.

## Case study data and image scale

Use the same fixed 4,096 CIFAR-10 images for comparable examples. Faces are naturally interpretable, but CIFAR-10 avoids external dataset setup.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, utils
import matplotlib.pyplot as plt
torch.manual_seed(7)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
tf=transforms.Compose([transforms.Resize(32),transforms.ToTensor(),transforms.Normalize((.5,)*3,(.5,)*3)])
ds=datasets.CIFAR10("./data",train=True,download=True,transform=tf)
loader=DataLoader(Subset(ds,range(4096)),batch_size=64,shuffle=True,num_workers=0,drop_last=True)
print(DEVICE,len(loader.dataset))

## Mapping network and modulated convolution

In [ ]:
Z_DIM=W_DIM=128
class Mapping(nn.Module):
    def __init__(self):
        super().__init__(); layers=[]
        for i in range(4): layers += [nn.Linear(Z_DIM if i==0 else W_DIM,W_DIM),nn.LeakyReLU(.2)]
        self.net=nn.Sequential(*layers)
    def forward(self,z): return self.net(F.normalize(z,dim=1))
class ModConv(nn.Module):
    def __init__(self,ci,co,k=3):
        super().__init__(); self.ci=ci; self.co=co; self.k=k; self.weight=nn.Parameter(torch.randn(1,co,ci,k,k)); self.affine=nn.Linear(W_DIM,ci); nn.init.ones_(self.affine.bias)
    def forward(self,x,w):
        b,_,h,ww=x.shape; wt=self.weight*self.affine(w).view(b,1,self.ci,1,1)
        wt=wt*torch.rsqrt(wt.square().sum((2,3,4),keepdim=True)+1e-8)
        y=F.conv2d(x.reshape(1,b*self.ci,h,ww),wt.reshape(b*self.co,self.ci,self.k,self.k),padding=self.k//2,groups=b)
        return y.reshape(b,self.co,h,ww)

## Compact style synthesis network

This teaching version reuses one W vector per image. Full StyleGAN has distinct style inputs per convolution and ToRGB operation, enabling true coarse-to-fine style mixing.

In [ ]:
class StyleBlock(nn.Module):
    def __init__(self,ci,co):
        super().__init__(); self.a=ModConv(ci,co); self.b=ModConv(co,co); self.na=nn.Parameter(torch.zeros(1,co,1,1)); self.nb=nn.Parameter(torch.zeros(1,co,1,1))
    def forward(self,x,w):
        x=F.interpolate(x,scale_factor=2,mode="nearest"); x=F.leaky_relu(self.a(x,w)+torch.randn_like(x[:,:1])*self.na,.2)
        return F.leaky_relu(self.b(x,w)+torch.randn_like(x[:,:1])*self.nb,.2)
class StyleGenerator(nn.Module):
    def __init__(self):
        super().__init__(); self.mapping=Mapping(); self.const=nn.Parameter(torch.randn(1,256,4,4)); self.start=ModConv(256,256)
        self.blocks=nn.ModuleList([StyleBlock(256,128),StyleBlock(128,64),StyleBlock(64,32)]); self.rgb=nn.ModuleList([nn.Conv2d(c,3,1) for c in (256,128,64,32)])
    def forward(self,z):
        w=self.mapping(z); x=F.leaky_relu(self.start(self.const.expand(len(z),-1,-1,-1),w),.2)
        for block in self.blocks: x=block(x,w)
        return torch.tanh(self.rgb[-1](x)),w
G=StyleGenerator().to(DEVICE); print(G(torch.randn(2,128,device=DEVICE))[0].shape)

## Style mixing and truncation

Truncation moves W toward its running mean: $w'=ar w+\psi(w-ar w)$. Lower $\psi$ often raises typical fidelity while reducing variety. This shared-style model cannot demonstrate actual layer-wise edits, but the helper describes the data flow.

In [ ]:
def truncate(w,w_avg,psi=.7): return w_avg+psi*(w-w_avg)
def mix_styles(w1,w2,num_layers,cutoff): return [w1 if i<cutoff else w2 for i in range(num_layers)]
G=StyleGenerator().to(DEVICE); w1=G.mapping(torch.randn(4,128,device=DEVICE)); w2=G.mapping(torch.randn(4,128,device=DEVICE))
print(len(mix_styles(w1,w2,8,3)),truncate(w1,w1.mean(0)).shape)

## Path length regularization

Random image-space noise probes the Jacobian from W to image. The squared difference between gradient norms and a moving target is the regularizer. It requires higher-order gradients and is usually applied lazily.

In [ ]:
def path_length_penalty(images,w,mean_length=0.,decay=.99):
    noise=torch.randn_like(images)/(images.shape[-1]*images.shape[-2])**.5
    grad=torch.autograd.grad((images*noise).sum(),w,create_graph=True)[0]
    lengths=(grad.square().sum(1)+1e-8).sqrt(); mean=decay*mean_length+(1-decay)*lengths.mean().detach()
    return (lengths-mean).square().mean(),mean
w=torch.randn(2,128,device=DEVICE,requires_grad=True); probe=w[:,0,None,None,None]*torch.ones(2,3,8,8,device=DEVICE)
print(path_length_penalty(probe,w)[0])

## Logistic adversarial update and StyleGAN2 context

A faithful StyleGAN2 run also needs filtered resampling, R1 regularization on real images, path length regularization, lazy regularization, and careful optimizer settings. This update only checks mechanics.

In [ ]:
class Disc(nn.Module):
    def __init__(self):
        super().__init__(); self.net=nn.Sequential(nn.Conv2d(3,64,4,2,1),nn.LeakyReLU(.2),nn.Conv2d(64,128,4,2,1),nn.LeakyReLU(.2),nn.Conv2d(128,256,4,2,1),nn.LeakyReLU(.2),nn.Flatten(),nn.Linear(4096,1))
    def forward(self,x): return self.net(x)
D=Disc().to(DEVICE); optD=torch.optim.Adam(D.parameters(),2e-4,betas=(0,.99)); optG=torch.optim.Adam(G.parameters(),2e-4,betas=(0,.99))
real,_=next(iter(loader)); real=real.to(DEVICE); b=len(real); fake=G(torch.randn(b,128,device=DEVICE))[0].detach()
ld=F.softplus(-D(real)).mean()+F.softplus(D(fake)).mean(); optD.zero_grad(); ld.backward(); optD.step()
lg=F.softplus(-D(G(torch.randn(b,128,device=DEVICE))[0])).mean(); optG.zero_grad(); lg.backward(); optG.step(); print(float(ld),float(lg))

## Latent interpolation and comparison

Smooth interpolation finds artifacts but does not prove disentanglement. Fixed noise helps isolate styles from stochastic texture. ProGAN is a resolution curriculum; StyleGAN emphasizes layer-wise synthesis; StyleGAN2 improves signal behavior and artifact quality.

In [ ]:
def slerp(a,b,t):
    a=F.normalize(a,dim=-1); b=F.normalize(b,dim=-1); om=torch.acos((a*b).sum(-1,keepdim=True).clamp(-.999,.999))
    return torch.sin((1-t)*om)/torch.sin(om)*a+torch.sin(t*om)/torch.sin(om)*b
G.eval()
with torch.no_grad():
    a=torch.randn(1,128,device=DEVICE); b=torch.randn(1,128,device=DEVICE); imgs=torch.cat([G(slerp(a,b,t))[0] for t in torch.linspace(0,1,8,device=DEVICE)]).cpu()
plt.figure(figsize=(12,2)); plt.axis("off"); plt.imshow(utils.make_grid(imgs,nrow=8,normalize=True).permute(1,2,0));

## Key Takeaways

- The mapping network transforms Z into W; per-layer affine styles steer synthesis at multiple scales.
- Noise injection adds localized stochastic detail.
- StyleGAN2 uses weight modulation/demodulation to control activation scale and reduce artifacts.
- Path length regularization targets consistent sensitivity to W movement.
- Truncation trades diversity for typicality; style mixing trains layer flexibility.
- Reproduction quality needs more data, resolution, compute, and optimized kernels.